In [ ]:
# --- Final Suite Notebook (v4): single + multi + storytelling table (plug-and-play) ---
%matplotlib widget
import sys
import logging
from pathlib import Path

import pandas as pd
from IPython.display import display

from forecast_pipeline.io_utils import configure_logging

configure_logging()
log = logging.getLogger("FINAL SUITE NOTEBOOK")

# =============================================================================
# 1) Resolve repo root + sys.path
# =============================================================================
try:
    # Jupyter path heuristic (keeps compatibility with your current notebook layout)
    REPO_ROOT = Path(get_ipython().run_line_magic("pwd", "")).resolve()
    if "notebooks" in REPO_ROOT.parts:
        notebooks_idx = REPO_ROOT.parts.index("notebooks")
        REPO_ROOT = Path(*REPO_ROOT.parts[:notebooks_idx])
except Exception:
    REPO_ROOT = Path.cwd().parent.parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

log.info("Repo root: %s", REPO_ROOT)

# =============================================================================
# 2) Engine imports (final suite + storytelling transformation)
# =============================================================================
from hpo.final_suite import (
    run_final_suite_single,
    run_final_suite_multi,
    build_experiment_story_table,
    resolve_arps_pure_experiments_from_positions,
)

# =============================================================================
# 3) Experiment campaign groups (split -> campaigns)
#    IMPORTANT: keep this dict as the source of truth for split membership.
# =============================================================================
CAMPAIGN_SPLITS = {
    "split_0.40": [
        "HPO_100_Lag_100_Horizon_150",
        "HPO_54_Lag_100_Horizon_150",
        "HPO_150_Lag_100_Horizon_150",
    ],
    "split_0.45": [
        "HPO_105_Lag_100_Horizon_150",
        "HPO_60_Lag_100_Horizon_150",
        "HPO_153_Lag_100_Horizon_150",
    ],
    "split_0.50": [
        "HPO_110_Lag_100_Horizon_150",
        "HPO_64_Lag_100_Horizon_150",
        "HPO_156_Lag_100_Horizon_150",
    ],
    "split_0.55": [
        "HPO_115_Lag_100_Horizon_150",
        "HPO_72_Lag_100_Horizon_150",
        "HPO_159_Lag_100_Horizon_150",
    ],
}

# Add a convenience aggregate group without polluting the split mapping used by the transformer
EXPERIMENT_GROUPS = {
    **CAMPAIGN_SPLITS,
    "all": [exp for exps in CAMPAIGN_SPLITS.values() for exp in exps],
}

# =============================================================================
# 4) Storytelling mapping configuration (self-explanatory)
# =============================================================================
# For each split, define which campaign position corresponds to ARPS Pure.
# - Position 0 = first experiment in that split list
# - None = ARPS Pure not added yet for this split
ARPS_PURE_POSITION_BY_SPLIT = {
    "split_0.40": 0,   # HPO_100... is the ARPS Pure campaign in split 0.40
    "split_0.45": 0,   # HPO_105
    "split_0.50": 0,   # HPO_110
    "split_0.55": 0,   # HPO_115
}

ARPS_PURE_EXP_BY_SPLIT = resolve_arps_pure_experiments_from_positions(
    campaign_splits=CAMPAIGN_SPLITS,
    pure_position_by_split=ARPS_PURE_POSITION_BY_SPLIT,
    logger=log,
)

log.info("ARPS Pure mapping by split: %s", ARPS_PURE_EXP_BY_SPLIT)

# =============================================================================
# 5) User controls
# =============================================================================
RUN_MODE = "multi"  # Options: "single" | "multi"

SINGLE_EXPERIMENT = "HPO_64_Lag_100_Horizon_150"

# If RUN_MODE == "multi", choose one group:
# "split_0.40", "split_0.45", "split_0.50", "split_0.55", or "all"
SELECTED_GROUP = "all"

# =============================================================================
# 6) Final suite execution options (kept close to your current defaults)
# =============================================================================
TOP_N = 2
SORT_METRIC = "val_smape_agg"
INCLUDE_TEST_METRICS = True
APPLY_LABEL_MAP = True
TOPN_PER_ARCHITECTURE = True

# Toggle storytelling table rendering (original table is always preserved/displayed)
SHOW_STORY_TABLE = True

# =============================================================================
# 7) Resolve execution targets
# =============================================================================
if RUN_MODE not in {"single", "multi"}:
    raise ValueError("RUN_MODE must be 'single' or 'multi'.")

if RUN_MODE == "single":
    selected_experiment = SINGLE_EXPERIMENT
    print(f"Mode: {RUN_MODE.upper()} | Selected experiment: {selected_experiment}")
else:
    if SELECTED_GROUP not in EXPERIMENT_GROUPS:
        raise KeyError(
            f"SELECTED_GROUP='{SELECTED_GROUP}' not found. "
            f"Available: {list(EXPERIMENT_GROUPS.keys())}"
        )
    experiments_to_run = EXPERIMENT_GROUPS[SELECTED_GROUP]
    print(f"Mode: {RUN_MODE.upper()} | Selected group: {SELECTED_GROUP}")
    print(f"Total experiments to run: {len(experiments_to_run)}")
    # print(experiments_to_run)

# =============================================================================
# 8) Execute: single
# =============================================================================
if RUN_MODE == "single":
    res = run_final_suite_single(
        REPO_ROOT,
        selected_experiment,
        top_n=TOP_N,
        sort_metric=SORT_METRIC,
        include_test_metrics=INCLUDE_TEST_METRICS,
        apply_label_map=APPLY_LABEL_MAP,
        topn_per_architecture=TOPN_PER_ARCHITECTURE,
    )

    table1 = res["table1"]
    table2 = res["table2"]

    log.info(
        "SINGLE: exp=%s | table1=%s | table2=%s",
        selected_experiment,
        getattr(table1, "shape", None),
        getattr(table2, "shape", None),
    )

    display(table1)
    display(table2)

    # Optional: storytelling transform also works in single mode
    if SHOW_STORY_TABLE and isinstance(table2, pd.DataFrame):
        table2_story = build_experiment_story_table(
            table2,
            campaign_splits=CAMPAIGN_SPLITS,           # split context lookup
            arps_pure_exp_by_split=ARPS_PURE_EXP_BY_SPLIT,
            logger=log,
        )
        display(table2_story)

# =============================================================================
# 9) Execute: multi
# =============================================================================
elif RUN_MODE == "multi":
    res = run_final_suite_multi(
        REPO_ROOT,
        experiments_to_run,
        top_n=TOP_N,
        sort_metric=SORT_METRIC,
        include_test_metrics=INCLUDE_TEST_METRICS,
        apply_label_map=APPLY_LABEL_MAP,
        topn_per_architecture=TOPN_PER_ARCHITECTURE,
        # prefer_metric_for_order="test_smape_agg",     # optional
        # fallback_metric_for_order="val_smape_agg",    # optional
        # dominant_threshold=0.80,                      # optional
    )

    table2_unified = res["table2_unified"]
    tables2_by_exp = res["tables2_by_exp"]
    tables1_by_exp = res["tables1_by_exp"]  # available, but not displayed by default in multi mode

    log.info(
        "MULTI: exps=%d | unified=%s | tables2_by_exp=%d | tables1_by_exp=%d",
        len(experiments_to_run),
        getattr(table2_unified, "shape", None),
        len(tables2_by_exp),
        len(tables1_by_exp),
    )

    # 1) Original unified table (unchanged)
    display(table2_unified)

    # 2) Storytelling / self-contained table (derived)
    if SHOW_STORY_TABLE and isinstance(table2_unified, pd.DataFrame):
        table2_unified_story = build_experiment_story_table(
            table2_unified,
            campaign_splits=CAMPAIGN_SPLITS,           # use only real split groups (no "all")
            arps_pure_exp_by_split=ARPS_PURE_EXP_BY_SPLIT,
            logger=log,
        )
        display(table2_unified_story)


    from forecast_pipeline._plotting_core import plot_final_suite_storyboard, plot_unified_story_winner_dashboard
    out = plot_unified_story_winner_dashboard(table2_unified_story, palette="default")

    out = plot_final_suite_storyboard(
        table2_unified_story,
        title_prefix="Results",
    
        # fontes grandes, porém controladas
        note_fontsize_plot1=14,   # no Plot 1 precisa ser menor para não colidir
        note_fontsize_plot2=15,
        badge_fontsize=15,
        trial_label_fontsize=16,
        split_title_fontsize=19,
        axis_label_fontsize=15,
        tick_fontsize=13,
        panel_title_fontsize=17,
        suptitle_fontsize=20,
        legend_fontsize_plot1=13,
        legend_fontsize_plot2=13,
    
        # layout
        split_title_x=0.16,         # empurra "Split 0.xx" para a direita
        plot1_figsize=(14.5, 6.4),
        plot2_fig_width=14.5,
        plot2_fig_row_height=4.9,
        dpi=120,
    
        # legendas quebradas (evita sobreposição)
        plot1_legend_ncol=3,
        plot2_legend_ncol=3,
    
        # posicionamento das notas
        plot1_left_note_loc="top-left",
        plot1_right_note_loc="bottom-right",  # crucial para evitar conflito no topo
        plot2_note_loc="bottom-right",
    )

    winner_map = build_split_winner_map_table(
        tables=tables1_by_exp,   # ou tables2_by_exp, desde que seja [(exp_name, df), ...]
        campaign_splits=CAMPAIGN_SPLITS,
        arps_pure_exp_by_split=ARPS_PURE_EXP_BY_SPLIT,
        proposed_budget_policy="min_budget",   # <- usa a campanha menor dos métodos propostos
    )
    
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(winner_map["compact_table"])

    # Optional: inspect per-campaign tables
    for exp_name, t2 in tables1_by_exp:
        print(f"\n==== {exp_name} ====")
        display(t2)


In [ ]:
from __future__ import annotations

import re
from typing import Mapping, Sequence, Tuple, Union, Optional
import pandas as pd


def _normalize_campaign_tables(
    tables: Union[pd.DataFrame, Mapping[str, pd.DataFrame], Sequence[Tuple[str, pd.DataFrame]]],
    campaign_splits: Mapping[str, Sequence[str]],
    experiment_col: str = "Experiment",
    split_col: str = "Split",
) -> pd.DataFrame:
    """
    Normalize campaign tables into a single DataFrame with Experiment and Split columns.
    """
    exp_to_split = {
        exp_name: split_name
        for split_name, exp_list in campaign_splits.items()
        for exp_name in exp_list
    }

    if isinstance(tables, pd.DataFrame):
        df = tables.copy()
        if experiment_col not in df.columns:
            raise KeyError(
                f"Unified DataFrame must contain '{experiment_col}'. "
                f"Available columns: {list(df.columns)}"
            )
    elif isinstance(tables, Mapping):
        parts = []
        for exp_name, t in tables.items():
            tmp = t.copy()
            tmp[experiment_col] = exp_name
            parts.append(tmp)
        df = pd.concat(parts, ignore_index=True)
    else:
        parts = []
        for exp_name, t in tables:
            tmp = t.copy()
            tmp[experiment_col] = exp_name
            parts.append(tmp)
        df = pd.concat(parts, ignore_index=True)

    df[split_col] = df[experiment_col].map(exp_to_split)
    if df[split_col].isna().any():
        missing = sorted(df.loc[df[split_col].isna(), experiment_col].dropna().unique().tolist())
        raise KeyError(
            "Some experiments could not be mapped to a split. "
            f"Missing in CAMPAIGN_SPLITS: {missing}"
        )

    # Preserve original row order (important: first row = validation-selected champion)
    df = df.copy()
    df["_row_order"] = range(len(df))
    return df


def _extract_budget(exp_name: str) -> int:
    m = re.search(r"HPO_(\d+)_", str(exp_name))
    if not m:
        raise ValueError(f"Could not extract HPO budget from experiment name: {exp_name}")
    return int(m.group(1))


def _classify_method(
    df: pd.DataFrame,
    arps_pure_exp_by_split: Mapping[str, Optional[str]],
    experiment_col: str = "Experiment",
    split_col: str = "Split",
    architecture_col: str = "Architecture",
    method_col: str = "Method",
) -> pd.DataFrame:
    """
    Add a high-level Method column:
      - Arps Pure
      - Arps Ensemble
      - PINN Analytical
    """
    pure_pairs = {
        (split_name, exp_name)
        for split_name, exp_name in arps_pure_exp_by_split.items()
        if exp_name is not None
    }

    def _row_method(row):
        split_name = row[split_col]
        exp_name = row[experiment_col]
        arch = str(row[architecture_col]).strip().upper()

        if (split_name, exp_name) in pure_pairs:
            return "Arps Pure"
        if arch == "PINN":
            return "PINN Analytical"
        if arch == "ARPS":
            return "Arps Ensemble"
        return arch.title()

    out = df.copy()
    out[method_col] = out.apply(_row_method, axis=1)
    out["Budget"] = out[experiment_col].map(_extract_budget)
    return out


def _resolve_selected_budgets(
    campaign_splits: Mapping[str, Sequence[str]],
    arps_pure_exp_by_split: Mapping[str, Optional[str]],
    proposed_budget_policy: Union[str, Mapping[str, int]] = "min_budget",
) -> pd.DataFrame:
    """
    Resolve which campaign budget will be used per split.

    Policies:
      - 'min_budget' : use the smaller proposed budget in each split
      - 'max_budget' : use the larger proposed budget in each split
      - dict         : explicit mapping, e.g. {'split_0.40': 54, ...}
    """
    rows = []

    for split_name, exp_list in campaign_splits.items():
        pure_exp = arps_pure_exp_by_split.get(split_name)
        pure_budget = _extract_budget(pure_exp) if pure_exp is not None else None

        proposed_budgets = sorted(
            {_extract_budget(e) for e in exp_list if e != pure_exp}
        )

        if len(proposed_budgets) == 0:
            selected_proposed_budget = None
        elif isinstance(proposed_budget_policy, Mapping):
            selected_proposed_budget = proposed_budget_policy[split_name]
        elif proposed_budget_policy == "min_budget":
            selected_proposed_budget = min(proposed_budgets)
        elif proposed_budget_policy == "max_budget":
            selected_proposed_budget = max(proposed_budgets)
        else:
            raise ValueError(
                "proposed_budget_policy must be 'min_budget', 'max_budget', "
                "or a dict mapping split -> budget"
            )

        rows.append({
            "Split": split_name,
            "Arps Pure budget": pure_budget,
            "Selected proposed budget": selected_proposed_budget,
        })

    return pd.DataFrame(rows)


def _select_campaign_rows(
    raw: pd.DataFrame,
    selected_budgets: pd.DataFrame,
    *,
    arps_pure_exp_by_split: Mapping[str, Optional[str]],
    split_col: str = "Split",
    experiment_col: str = "Experiment",
    method_col: str = "Method",
) -> pd.DataFrame:
    """
    Keep only:
      - the Arps Pure campaign for each split
      - the selected proposed budget for Arps Ensemble and PINN Analytical
    """
    budget_map = dict(
        zip(selected_budgets["Split"], selected_budgets["Selected proposed budget"])
    )

    def _keep(row):
        split_name = row[split_col]
        exp_name = row[experiment_col]
        method = row[method_col]
        budget = row["Budget"]

        if method == "Arps Pure":
            return exp_name == arps_pure_exp_by_split.get(split_name)

        # proposed methods
        return budget == budget_map.get(split_name)

    out = raw[raw.apply(_keep, axis=1)].copy()
    return out


def _pick_validation_champion_per_method_well(
    df: pd.DataFrame,
    *,
    split_col: str = "Split",
    dataset_col: str = "Dataset",
    well_col: str = "Well",
    method_col: str = "Method",
) -> pd.DataFrame:
    """
    Within the selected campaign rows, keep the first row per
    (split, dataset, well, method), assuming the input table is already
    ordered by validation ranking (as in your final suite notebook output).
    """
    key_cols = [split_col, dataset_col, well_col, method_col]
    out = (
        df.sort_values("_row_order")
          .drop_duplicates(subset=key_cols, keep="first")
          .copy()
    )
    return out


def _abbreviate_well(dataset: str, well: str) -> str:
    dataset = str(dataset)
    well = str(well)

    if dataset.upper() == "VOLVE":
        if "15/9-F-12" in well:
            return "F12"
        if "15/9-F-14" in well:
            return "F14"
    return well


def _winner_string_for_metric(
    df: pd.DataFrame,
    *,
    metric_col: str,
    split_col: str = "Split",
    dataset_col: str = "Dataset",
    well_col: str = "Well",
    method_col: str = "Method",
) -> pd.DataFrame:
    """
    Build per-split compact winner strings for one metric.
    """
    method_order = ["PINN Analytical", "Arps Ensemble", "Arps Pure"]

    ranked = (
        df.sort_values([split_col, dataset_col, well_col, metric_col, method_col])
          .copy()
    )

    ranked["rank"] = (
        ranked.groupby([split_col, dataset_col, well_col]).cumcount() + 1
    )
    winners = ranked[ranked["rank"] == 1].copy()

    winners["WellShort"] = winners.apply(
        lambda r: _abbreviate_well(r[dataset_col], r[well_col]), axis=1
    )

    rows = []
    for split_name, split_df in winners.groupby(split_col, sort=True):
        split_df = split_df.sort_values([dataset_col, well_col])

        parts = []
        for method in method_order:
            wells = split_df.loc[split_df[method_col] == method, "WellShort"].tolist()
            if wells:
                parts.append(f"{method} ({len(wells)}): {', '.join(wells)}")

        rows.append({
            "Split": split_name,
            "Winner map": "; ".join(parts),
        })

    return pd.DataFrame(rows)


def build_split_winner_map_table(
    tables: Union[pd.DataFrame, Mapping[str, pd.DataFrame], Sequence[Tuple[str, pd.DataFrame]]],
    *,
    campaign_splits: Mapping[str, Sequence[str]],
    arps_pure_exp_by_split: Mapping[str, Optional[str]],
    proposed_budget_policy: Union[str, Mapping[str, int]] = "min_budget",
    experiment_col: str = "Experiment",
    split_col: str = "Split",
    dataset_col: str = "Dataset",
    well_col: str = "Well",
    architecture_col: str = "Architecture",
    rate_col: str = "Production Rate (Test)",
    cum_col: str = "Cumulative Production (Test)",
    method_col: str = "Method",
) -> dict:
    """
    Main function.

    Returns:
      - compact_table: final compact summary for the paper
      - selected_budgets: which budgets were used per split
      - selected_rows: rows retained after campaign selection
      - champions: one validation-selected champion per method/well/split
    """
    raw = _normalize_campaign_tables(
        tables=tables,
        campaign_splits=campaign_splits,
        experiment_col=experiment_col,
        split_col=split_col,
    )

    raw = _classify_method(
        raw,
        arps_pure_exp_by_split=arps_pure_exp_by_split,
        experiment_col=experiment_col,
        split_col=split_col,
        architecture_col=architecture_col,
        method_col=method_col,
    )

    selected_budgets = _resolve_selected_budgets(
        campaign_splits=campaign_splits,
        arps_pure_exp_by_split=arps_pure_exp_by_split,
        proposed_budget_policy=proposed_budget_policy,
    )

    selected_rows = _select_campaign_rows(
        raw,
        selected_budgets,
        arps_pure_exp_by_split=arps_pure_exp_by_split,
        split_col=split_col,
        experiment_col=experiment_col,
        method_col=method_col,
    )

    champions = _pick_validation_champion_per_method_well(
        selected_rows,
        split_col=split_col,
        dataset_col=dataset_col,
        well_col=well_col,
        method_col=method_col,
    )

    rate_map = _winner_string_for_metric(
        champions,
        metric_col=rate_col,
        split_col=split_col,
        dataset_col=dataset_col,
        well_col=well_col,
        method_col=method_col,
    ).rename(columns={"Winner map": "Rate winner map"})

    cum_map = _winner_string_for_metric(
        champions,
        metric_col=cum_col,
        split_col=split_col,
        dataset_col=dataset_col,
        well_col=well_col,
        method_col=method_col,
    ).rename(columns={"Winner map": "Cumulative winner map"})

    compact = (
        selected_budgets.merge(rate_map, on="Split", how="left")
                        .merge(cum_map, on="Split", how="left")
                        .copy()
    )

    compact["Split"] = compact["Split"].str.replace("split_", "", regex=False)

    return {
        "compact_table": compact,
        "selected_budgets": selected_budgets,
        "selected_rows": selected_rows,
        "champions": champions,
    }

In [ ]:
import re
import pandas as pd

def make_consolidated_table_latex(
    table1: pd.DataFrame,
    table2: pd.DataFrame,  # <- vindo do build_architecture_summary_iqr(...)
    caption="Per-well results (Panel A) and architecture-level summary (Panel B). Lower SMAPE (\\%) is better.",
    label="tab:consolidated",
    shade_architecture: bool = True,
    arch_highlight_color: str = "black!12",
    bold_architecture: bool = True,
) -> str:
    """
    Gera a tabela LaTeX consolidada (Panel A + Panel B).
    Panel A: igual (Means por linha, destaques, etc.)
    Panel B: MEAN (numérico) + IQR no formato textual 'q1–q3', SEM converter para float.
             As células de IQR usam \multicolumn{1}{c}{...} para sobrepor o S[...] e aceitar texto.
    """

    numeric_cols = table2.select_dtypes(include=np.number).columns

    # 2. Aplica o arredondamento a essas colunas
    table2[numeric_cols] = table2[numeric_cols].round(2)

    # ------------------ helpers ------------------
    def esc(s):
        if pd.isna(s):
            return ""
        s = str(s)
        repl = {
            "&": r"\&", "%": r"\%", "#": r"\#", "$": r"\$",
            "{": r"\{", "}": r"\}", "_": r"\_", "~": r"\textasciitilde{}",
            "^": r"\textasciicircum{}", "\\": r"\textbackslash{}",
        }
        for k, v in repl.items():
            s = s.replace(k, v)
        return s

    def num(x):
        # mantemos 6 casas; arredondamento final fica por conta do siunitx via S[table-format]
        if pd.isna(x):
            return r"\num{0}"
        return rf"\num{{{float(x):.2f}}}"

    def pretty_dataset(name: str) -> str:
        if pd.isna(name):
            return ""
        return str(name).replace("_", " ")

    def iqr_text_cell(v) -> str:
        """
        Retorna \multicolumn{1}{c}{<texto IQR>} preservando o intervalo textual (ex.: 6.39–9.28).
        - Normaliza travessões: garante '–' (en dash) no resultado.
        - Faz escapings LaTeX.
        """
        if pd.isna(v):
            txt = ""
        else:
            s = str(v).strip()
            # normaliza para en dash
            s = s.replace("—", "–").replace("--", "–")
            # remove espaços exagerados ao redor do dash
            s = re.sub(r"\s*–\s*", "–", s)
            txt = esc(s)
        return rf"\multicolumn{{1}}{{c}}{{{txt}}}"

    arch_order = ["PINN", "ARPS", "DARTS"]

    # ------------------ Panel A (Table 1) ------------------
    t1 = table1.rename(columns={
        "Production Rate (Validation)": "val_rate",
        "Cumulative Production (Validation)": "val_cum",
        "Production Rate (Test)": "test_rate",
        "Cumulative Production (Test)": "test_cum",
    }).copy()

    t1["arch_rank"] = t1["Architecture"].apply(
        lambda a: arch_order.index(a) if a in arch_order else len(arch_order)
    )
    t1 = t1.sort_values(by=["Dataset", "Well", "arch_rank"]).reset_index(drop=True)

    panelA_rows = []
    for (ds, well), grp in t1.groupby(["Dataset", "Well"], sort=False):
        idx_best_rate = grp["test_rate"].astype(float).idxmin()
        idx_best_cum  = grp["test_cum"].astype(float).idxmin()

        first = True
        for ridx, row in grp.iterrows():
            ds_cell   = esc(pretty_dataset(ds)) if first else ""
            well_cell = esc(well) if first else ""

            arch_text = esc(row["Architecture"])
            is_winner_line = (ridx == idx_best_rate) or (ridx == idx_best_cum)
            if is_winner_line:
                if shade_architecture and bold_architecture:
                    arch_cell = rf"\cellcolor{{{arch_highlight_color}}}\textbf{{{arch_text}}}"
                elif shade_architecture:
                    arch_cell = rf"\cellcolor{{{arch_highlight_color}}}{arch_text}"
                elif bold_architecture:
                    arch_cell = rf"\textbf{{{arch_text}}}"
                else:
                    arch_cell = arch_text
            else:
                arch_cell = arch_text

            strat = esc(row["Strategy"])
            reco  = esc(row["Reconstruction"])

            v_rate = num(row["val_rate"])
            v_cum  = num(row["val_cum"])

            if ridx == idx_best_rate:
                t_rate = rf"\bestA{{{float(row['test_rate']):.2f}}}"
            else:
                t_rate = num(row["test_rate"])

            if ridx == idx_best_cum:
                t_cum = rf"\bestA{{{float(row['test_cum']):.2f}}}"
            else:
                t_cum = num(row["test_cum"])

            line = f"{ds_cell} & {well_cell} & {arch_cell} & {strat} & {reco} & {v_rate} & {v_cum} & {t_rate} & {t_cum} \\\\"
            panelA_rows.append(line)
            first = False

    # ------------------ Panel B (Table 2: MEANS + IQR textual) ------------------
    mean_cols = [
        ("Production Rate (Validation) (MEAN)", "Rate (Val)"),
        ("Cumulative Production (Validation) (MEAN)", "Cum. (Val)"),
        ("Production Rate (Test) (MEAN)", "Rate (Test)"),
        ("Cumulative Production (Test) (MEAN)", "Cum. (Test)"),
    ]
    iqr_cols = [
        ("Production Rate (Validation) (IQR)", "Rate (Val)"),
        ("Cumulative Production (Validation) (IQR)", "Cum. (Val)"),
        ("Production Rate (Test) (IQR)", "Rate (Test)"),
        ("Cumulative Production (Test) (IQR)", "Cum. (Test)"),
    ]

    # vencedores por média (menor mean)
    best_mean = {}
    for col, _ in mean_cols:
        if col in table2.columns:
            col_min_idx = table2[col].astype(float).idxmin()
            best_mean[col] = str(table2.loc[col_min_idx, "Architecture"])

    b2 = table2.copy()
    b2["arch_rank"] = b2["Architecture"].apply(
        lambda a: arch_order.index(a) if a in arch_order else len(arch_order)
    )
    b2 = b2.sort_values(by=["arch_rank", "Architecture"])

    panelB_rows = []
    for _, row in b2.iterrows():
        arch = esc(row["Architecture"])
        mean_cells = []
        for col, _ in mean_cols:
            if col not in row:
                continue
            val = float(row[col])
            if best_mean.get(col, None) == row["Architecture"]:
                mean_cells.append(rf"\bestB{{{val:.2f}}}")
            else:
                mean_cells.append(num(val))

        # IQRs como texto "q1–q3"
        iqr_cells = []
        for col, _ in iqr_cols:
            if col in row:
                iqr_cells.append(iqr_text_cell(row[col]))
        line = f"{arch} & " + " & ".join(mean_cells + iqr_cells) + r" \\"
        panelB_rows.append(line)

    # ------------------ LaTeX ------------------
    latex = rf"""
% --- TABELA (gerada automaticamente) ---
\begin{{table*}}[!htb]
\centering
\caption{{{caption}}}
\label{{{label}}}
\begin{{threeparttable}}
\resizebox{{\textwidth}}{{!}}{{%
\begin{{tabular}}{{%
  @{{}}l l l l l
  S[table-format=2.2]   % Val Rate
  S[table-format=2.2]   % Val Cum
  S[table-format=3.2]   % Test Rate (até ~140)
  S[table-format=2.2]   % Test Cum
  @{{}}
}}
\toprule
\multicolumn{{5}}{{c}}{{}} &
\multicolumn{{2}}{{c}}{{\textbf{{Validation}}}} &
\multicolumn{{2}}{{c}}{{\textbf{{Test}}}} \\
\cmidrule(lr){{6-7}}\cmidrule(lr){{8-9}}
\textbf{{Dataset}} & \textbf{{Well}} & \textbf{{Architecture}} & \textbf{{Strategy}} & \textbf{{Reconstruction}} &
{{\makecell{{\textbf{{Rate}}}}}} & {{\makecell{{\textbf{{Cum.}}}}}} &
{{\makecell{{\textbf{{Rate}}}}}} & {{\makecell{{\textbf{{Cum.}}}}}} \\
\midrule
\multicolumn{{9}}{{@{{}}l}}{{\textbf{{Panel A: Per-well showcase}}}}\\
\addlinespace[2pt]
{chr(10).join(panelA_rows)}
\midrule

\multicolumn{{9}}{{@{{}}l}}{{\textbf{{Panel B: Architecture-level summary (means and IQR across wells/datasets)}}}}\\
\addlinespace[2pt]

\textbf{{Architecture}} &
\multicolumn{{4}}{{c}}{{\textbf{{Means}}}} &
\multicolumn{{4}}{{c}}{{\textbf{{IQR (Q1–Q3, textual)}}}} \\
\cmidrule(lr){{2-5}}\cmidrule(lr){{6-9}}
& {{\makecell{{\textbf{{Rate}}\\\textbf{{(Val)}}}}}} &
  {{\makecell{{\textbf{{Cum.}}\\\textbf{{(Val)}}}}}} &
  {{\makecell{{\textbf{{Rate}}\\\textbf{{(Test)}}}}}} &
  {{\makecell{{\textbf{{Cum.}}\\\textbf{{(Test)}}}}}} &
  {{\makecell{{\textbf{{Rate}}\\\textbf{{(Val)}}}}}} &
  {{\makecell{{\textbf{{Cum.}}\\\textbf{{(Val)}}}}}} &
  {{\makecell{{\textbf{{Rate}}\\\textbf{{(Test)}}}}}} &
  {{\makecell{{\textbf{{Cum.}}\\\textbf{{(Test)}}}}}} \\
\addlinespace[2pt]
{chr(10).join(panelB_rows)}

\bottomrule
\end{{tabular}}}}
\begin{{tablenotes}}[flushleft]\footnotesize
\item \textbf{{Panel A.}} For each well, the lowest \emph{{Test Rate}} and \emph{{Test Cum.}} are shown in bold with \circledgold{{1}}; the winning row(s) also gray-highlight the \emph{{Architecture}} cell.
\item \textbf{{Panel B.}} Architecture-level \emph{{Means}} and textual \emph{{IQRs}} (Q1–Q3) across all wells/datasets; the lowest mean per metric is highlighted in bold with \circledgold{{1}}.
\end{{tablenotes}}
\end{{threeparttable}}
\end{{table*}}
""".strip()

    return latex


In [ ]:
latex_str = make_consolidated_table_latex(table1, table2)
print(latex_str)   # copie e cole no Overleaf

In [ ]:
# """
# ROBUST CHAMPION SELECTOR (VAL-only) + AUDIT (TEST-only) — Refactored (Compact, Plug-and-play)

# Goals:
# - Keep the SAME selection logic as the original script:
#   - Pool methods: TOP_PCT or VAL_BAND (VAL-only)
#   - Robust score: local neighborhood (within strategy if enough points else global),
#     feature-distance on [log10(lr), norm(epochs), norm(batch)], with weighted L1,
#     fallback to |VAL diff| when features invalid.
#   - Optional cycle-consistency (VAL-only): top-k hits across cycles using candidate_key.
#   - TEST is audit-only (regret/ratio/spearman/pool stats).

# How to use:
#   1) Edit CAMPAIGN_DIR
#   2) Run this file
# """

# from __future__ import annotations

# import math
# import os
# import re
# from dataclasses import dataclass
# from pathlib import Path
# from typing import Dict, List, Optional, Tuple

# import numpy as np
# import pandas as pd

# # Optional plotting integration (your framework)
# try:
#     from forecast_pipeline._plotting_core import _get_color_palette, plot_error_distributions_story
# except Exception:
#     _get_color_palette = None


# # =========================
# # CONFIG (EDIT THESE)
# # =========================
# CAMPAIGN_DIR = "/home/gabriel/Documentos/Equinor/src/experiment_configs/HPO_245_Lag_100_Horizon_150"
# RESULTS_SUBDIR = "results"

# OUTPUT_DIR = Path(CAMPAIGN_DIR) / "reports" / "robust_selection_reports"
# SAVE_MASTER = True
# SAVE_SUMMARY = True
# SAVE_DIAGNOSTICS = True

# MASTER_NAME = "master_leaderboard.csv"
# SUMMARY_NAME = "robust_summary.csv"
# DIAGNOSTICS_NAME = "leaderboards_diagnostics.csv"

# # Presentation
# PLOT = False
# MAX_GROUPS = 999

# # Pool selection (VAL-only)
# POOL_METHOD = "top_pct"  # "top_pct" or "val_band"
# POOL_CFG = dict(
#     top_pct=0.2,
#     drop=0.05,
#     take=0.40,
#     min_candidates=20,
# )

# # Robust score (VAL-only)
# ROBUST = dict(
#     k=10,
#     min_strat=25,
#     alpha=0.65,
#     beta=0.03,
#     gamma=0.35,
#     luck_q=0.25,
#     w_lr=2.0,
#     w_ep=0.25,
#     w_bs=0.50,
# )

# # Multi-cycle consistency (VAL-only)
# CONS = dict(
#     use=True,
#     topk_per_cycle=10,
#     min_hits=2,
#     min_rate=None,  # Optional[float]
#     stab_lambda=0.10,  # tie += lambda * IQR(val)
# )

# # Audit diagnostics (TEST-only)
# TEST_AUDIT = dict(
#     good_abs=None,  # Optional[float]
#     good_q=0.30,
# )

# # Within-leaderboard dedup (safe; does not dedup across cycles)
# DEDUP_WITHIN_KEYS = ("job_hash", "experiment_id")
# DEDUP_WITHIN_KEEP = "last"

# # Columns
# C = dict(
#     val="val_smape_agg",
#     val_cum="val_smape_cum",
#     test="test_smape_agg",  # audit-only
#     strat="physics_strategy",
#     trial="optuna_trial_number",
#     ep="epochs",
#     bs="batch_size",
#     lr="learning_rate",
#     cycle="cycle",
#     dataset="dataset",
#     well="well",
#     arch="architecture",
# )


# # =========================
# # Optional Rich output
# # =========================
# def _get_rich():
#     try:
#         from rich.console import Console
#         from rich.panel import Panel
#         from rich.table import Table

#         return Console(), Panel, Table
#     except Exception:
#         return None, None, None


# CONSOLE, RICH_PANEL, RICH_TABLE = _get_rich()


# # =========================
# # Small helpers
# # =========================
# def to_num(s: pd.Series) -> pd.Series:
#     return pd.to_numeric(s, errors="coerce").astype(float)


# def mad(x: np.ndarray) -> float:
#     x = x[np.isfinite(x)]
#     if len(x) == 0:
#         return np.nan
#     med = np.median(x)
#     return float(np.median(np.abs(x - med)))


# def iqr(x: np.ndarray) -> float:
#     x = x[np.isfinite(x)]
#     if len(x) == 0:
#         return np.nan
#     q75, q25 = np.percentile(x, [75, 25])
#     return float(q75 - q25)


# def rank_gap(df: pd.DataFrame) -> np.ndarray:
#     if C["val_cum"] not in df.columns:
#         return np.zeros(len(df), dtype=float)
#     a = to_num(df[C["val"]])
#     c = to_num(df[C["val_cum"]])
#     ra = a.rank(method="average", ascending=True)
#     rc = c.rank(method="average", ascending=True)
#     return (ra - rc).abs().to_numpy(dtype=float)


# def safe_read_csv(path: str) -> pd.DataFrame:
#     try:
#         return pd.read_csv(path, sep=",", encoding="utf-8-sig")
#     except Exception:
#         return pd.read_csv(path, sep=",", encoding="utf-8-sig", engine="python", on_bad_lines="warn")


# def _dedup_within(df: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
#     used = ""
#     for k in DEDUP_WITHIN_KEYS:
#         if k in df.columns:
#             used = k
#             break
#     if used:
#         df = df.drop_duplicates(subset=[used], keep=DEDUP_WITHIN_KEEP).reset_index(drop=True)
#     return df, used


# def short_family(dataset_name: str) -> str:
#     s = str(dataset_name).upper()
#     if s.startswith("VOLVE"):
#         return "VOLVE"
#     if s.startswith("UNISIM"):
#         return "UNISIM"
#     if "VALIDATION" in s:
#         return "VALIDATION"
#     return "OTHER"


# def _maybe_display_dataframe(df: pd.DataFrame) -> bool:
#     try:
#         from IPython.display import display  # type: ignore

#         display(df)
#         return True
#     except Exception:
#         return False


# # =========================
# # Discovery / Master builder (multi-cycle aware)
# # =========================
# CYCLE_RE = re.compile(r"(?P<prefix>.+)_(?P<arch>[^_]+)_cycle_(?P<cycle>\d+)$")


# def _parse_cycle_folder(folder_name: str) -> Optional[Dict[str, object]]:
#     m = CYCLE_RE.match(folder_name)
#     if not m:
#         return None

#     prefix = m.group("prefix")
#     arch = m.group("arch")
#     cycle = int(m.group("cycle"))

#     parts = prefix.split("_", 1)
#     dataset = parts[0] if parts else None
#     well_raw = parts[1] if len(parts) > 1 else None
#     well = well_raw.replace("_", "/") if isinstance(well_raw, str) else None

#     return dict(dataset=dataset, well=well, architecture=arch, cycle=cycle)


# def discover_leaderboards(base_dir: Path) -> List[Dict[str, object]]:
#     results_dir = base_dir / RESULTS_SUBDIR
#     if not results_dir.is_dir():
#         raise FileNotFoundError(f"results dir not found: {results_dir}")

#     out: List[Dict[str, object]] = []
#     for root, _, files in os.walk(results_dir):
#         if "robust_selection_reports" in root:
#             continue
#         if "leaderboard.csv" not in files:
#             continue

#         meta = _parse_cycle_folder(Path(root).name)
#         if meta is None:
#             continue

#         out.append(
#             dict(
#                 **meta,
#                 path=str(Path(root) / "leaderboard.csv"),
#                 root=str(root),
#             )
#         )

#     out.sort(key=lambda d: (str(d["dataset"]), str(d["well"]), str(d["architecture"]), int(d["cycle"])))
#     return out


# def build_master_leaderboard(base_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     lbs = discover_leaderboards(base_dir)
#     if not lbs:
#         raise RuntimeError(f"No *_cycle_* leaderboards found under: {base_dir / RESULTS_SUBDIR}")

#     frames: List[pd.DataFrame] = []
#     diag: List[Dict[str, object]] = []

#     for item in lbs:
#         df = safe_read_csv(item["path"])
#         n_raw = int(len(df))
#         df, used_key = _dedup_within(df)
#         n_after = int(len(df))

#         # attach meta safely
#         for k in ("dataset", "well", "architecture", "cycle"):
#             if k not in df.columns:
#                 df[k] = item[k]

#         if C["trial"] in df.columns:
#             df[C["trial"]] = pd.to_numeric(df[C["trial"]], errors="coerce")

#         diag.append(
#             dict(
#                 dataset=item["dataset"],
#                 well=item["well"],
#                 architecture=item["architecture"],
#                 cycle=int(item["cycle"]),
#                 n_rows_raw=n_raw,
#                 n_rows_after_within_dedup=n_after,
#                 within_dedup_key=used_key,
#                 leaderboard_path=item["path"],
#             )
#         )
#         frames.append(df)

#     master = pd.concat(frames, ignore_index=True, sort=False)
#     diag_df = pd.DataFrame(diag).sort_values(["dataset", "well", "architecture", "cycle"]).reset_index(drop=True)

#     title = f"Campaign: {base_dir}"
#     line = f"Leaderboards found (cycles): {len(diag_df)} | Master rows: {len(master)}"
#     if CONSOLE:
#         CONSOLE.print(RICH_PANEL(f"[bold]{title}[/bold]\n{line}", title="Discovery", expand=False))
#         cols = ["dataset", "well", "architecture", "cycle", "n_rows_raw", "n_rows_after_within_dedup", "within_dedup_key"]
#         CONSOLE.print(RICH_PANEL(diag_df[cols].to_string(index=False), title="Per-cycle leaderboards", expand=False))
#     else:
#         print("\n" + title)
#         print(line)
#         cols = ["dataset", "well", "architecture", "cycle", "n_rows_raw", "n_rows_after_within_dedup", "within_dedup_key"]
#         print(diag_df[cols].to_string(index=False))

#     return master, diag_df


# # =========================
# # Pool builder (VAL-only)
# # =========================
# def pick_pool_idx(df: pd.DataFrame) -> np.ndarray:
#     v = to_num(df[C["val"]])
#     ok = np.isfinite(v.to_numpy())
#     idx_all = df.index[ok]
#     if len(idx_all) == 0:
#         return np.array([], dtype=int)

#     idx_sorted = v.loc[idx_all].sort_values(ascending=True).index.to_numpy()
#     n = len(idx_sorted)

#     if POOL_METHOD.lower() == "top_pct":
#         k = min(n, max(POOL_CFG["min_candidates"], int(math.ceil(POOL_CFG["top_pct"] * n))))
#         return idx_sorted[:k].astype(int)

#     # val_band
#     drop_n = int(math.floor(POOL_CFG["drop"] * n))
#     take_n = max(POOL_CFG["min_candidates"], int(math.ceil(POOL_CFG["take"] * n)))
#     start = min(max(0, drop_n), n)
#     end = min(start + take_n, n)
#     if end <= start:
#         start, end = 0, min(POOL_CFG["min_candidates"], n)
#     return idx_sorted[start:end].astype(int)


# # =========================
# # Robust scoring engine (VAL-only) — SAME logic as original
# # =========================
# def _prep_numeric(df: pd.DataFrame) -> pd.DataFrame:
#     out = df.copy()
#     for col in (C["val"], C["val_cum"], C["test"], C["ep"], C["bs"], C["lr"], C["trial"]):
#         if col in out.columns:
#             out[col] = pd.to_numeric(out[col], errors="coerce")
#     if C["strat"] not in out.columns:
#         out[C["strat"]] = "unknown"
#     if C["trial"] not in out.columns:
#         out[C["trial"]] = np.nan
#     return out


# def _rob_norm_10_90(x: np.ndarray) -> np.ndarray:
#     x2 = x[np.isfinite(x)]
#     if len(x2) == 0:
#         return np.full_like(x, np.nan)
#     lo, hi = np.percentile(x2, [10, 90])
#     span = (hi - lo) if hi > lo else (np.std(x2) if np.std(x2) > 0 else 1.0)
#     return (x - lo) / (span + 1e-12)


# def _features(df: pd.DataFrame) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
#     need = (C["lr"], C["ep"], C["bs"])
#     if not all(c in df.columns for c in need):
#         return None, None

#     lr = df[C["lr"]].to_numpy(dtype=float)
#     lr = np.where(lr > 0, lr, np.nan)
#     log_lr = np.log10(lr)  # IMPORTANT: NOT normalized (matches original logic)

#     ep = _rob_norm_10_90(df[C["ep"]].to_numpy(dtype=float))
#     bs = _rob_norm_10_90(df[C["bs"]].to_numpy(dtype=float))

#     feats = np.column_stack([log_lr, ep, bs])
#     w = np.array([ROBUST["w_lr"], ROBUST["w_ep"], ROBUST["w_bs"]], dtype=float)
#     return feats, w


# def _l1_weighted(feats: np.ndarray, q: np.ndarray, w: np.ndarray) -> np.ndarray:
#     return np.sum(np.abs(feats - q[None, :]) * w[None, :], axis=1)


# def compute_robust_scores(df_in: pd.DataFrame, cand_idx: np.ndarray) -> Tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
#     """
#     Returns series aligned to cleaned+reset df:
#       scores, scopes, local_mads, rank_gaps
#     """
#     df = _prep_numeric(df_in)
#     df = df.loc[np.isfinite(df[C["val"]].to_numpy(dtype=float))].reset_index(drop=True)
#     n = len(df)

#     scores = pd.Series(np.nan, index=df.index, dtype=float)
#     scopes = pd.Series("", index=df.index, dtype=object)
#     local_mads = pd.Series(np.nan, index=df.index, dtype=float)
#     rgs = pd.Series(np.nan, index=df.index, dtype=float)

#     if n == 0 or len(cand_idx) == 0:
#         return scores, scopes, local_mads, rgs

#     rg = rank_gap(df)
#     feats, w = _features(df)

#     def neighbors(i: int) -> Tuple[np.ndarray, str]:
#         strat = df.loc[i, C["strat"]]
#         pool = df.index[df[C["strat"]] == strat].to_numpy()
#         scope = "within_strategy"
#         if len(pool) < ROBUST["min_strat"]:
#             pool = df.index.to_numpy()
#             scope = "global"

#         pool = pool[pool != i]
#         if len(pool) == 0:
#             return np.array([], dtype=int), scope

#         # fallback distance: |VAL diff|
#         if feats is None:
#             d = np.abs(df.loc[pool, C["val"]].to_numpy(dtype=float) - float(df.loc[i, C["val"]]))
#         else:
#             q = feats[i, :]
#             if not np.all(np.isfinite(q)):
#                 d = np.abs(df.loc[pool, C["val"]].to_numpy(dtype=float) - float(df.loc[i, C["val"]]))
#             else:
#                 pool_ok = pool[np.all(np.isfinite(feats[pool]), axis=1)]
#                 # If too few points are valid, fallback to global pool (matches original behavior)
#                 if len(pool_ok) < max(3, min(ROBUST["k"], len(pool))):
#                     if scope == "within_strategy":
#                         pool2 = df.index.to_numpy()
#                         pool2 = pool2[pool2 != i]
#                         pool_ok = pool2[np.all(np.isfinite(feats[pool2]), axis=1)]
#                         scope = "global"
#                 if len(pool_ok) == 0:
#                     d = np.abs(df.loc[pool, C["val"]].to_numpy(dtype=float) - float(df.loc[i, C["val"]]))
#                 else:
#                     pool = pool_ok
#                     d = _l1_weighted(feats[pool], q, w)

#         k = min(ROBUST["k"], len(pool))
#         nn = pool[np.argsort(d)[:k]]
#         return nn.astype(int), scope

#     for i in cand_idx.astype(int):
#         if i < 0 or i >= n:
#             continue

#         nn, scope = neighbors(int(i))
#         vals_nn = (
#             np.concatenate([[float(df.loc[i, C["val"]])], df.loc[nn, C["val"]].to_numpy(dtype=float)])
#             if len(nn)
#             else np.array([float(df.loc[i, C["val"]])], dtype=float)
#         )

#         loc_mad = mad(vals_nn)
#         if not np.isfinite(loc_mad) or loc_mad < 1e-12:
#             loc_mad = float(np.std(vals_nn, ddof=1)) if len(vals_nn) > 1 else 0.0

#         rg_i = float(rg[i]) if np.isfinite(rg[i]) else 0.0

#         # luckiness penalty
#         if len(nn) >= 3:
#             neigh_vals = df.loc[nn, C["val"]].to_numpy(dtype=float)
#             baseline = float(np.quantile(neigh_vals, ROBUST["luck_q"]))
#             spread = mad(neigh_vals)
#             if not np.isfinite(spread) or spread < 1e-12:
#                 spread = float(np.std(neigh_vals, ddof=1)) if len(neigh_vals) > 1 else 1.0
#             z = (baseline - float(df.loc[i, C["val"]])) / (spread + 1e-12)
#             luck = float(max(0.0, z))
#         else:
#             luck = 0.0

#         score = float(np.mean(vals_nn)) + ROBUST["alpha"] * float(loc_mad) + ROBUST["beta"] * float(rg_i) + ROBUST["gamma"] * float(luck)

#         scores.iloc[i] = score
#         scopes.iloc[i] = scope
#         local_mads.iloc[i] = float(loc_mad)
#         rgs.iloc[i] = float(rg_i)

#     return scores, scopes, local_mads, rgs


# # =========================
# # Selection (VAL-only)
# # =========================
# def _cand_key(row: pd.Series) -> str:
#     tr = row.get(C["trial"], np.nan)
#     if pd.notna(tr) and np.isfinite(float(tr)):
#         return f"trial:{int(float(tr))}"
#     if "job_hash" in row and pd.notna(row["job_hash"]):
#         return f"job:{str(row['job_hash'])}"
#     if "experiment_id" in row and pd.notna(row["experiment_id"]):
#         return f"exp:{str(row['experiment_id'])}"
#     s = str(row.get(C["strat"], "unknown"))
#     lr = row.get(C["lr"], np.nan)
#     ep = row.get(C["ep"], np.nan)
#     bs = row.get(C["bs"], np.nan)
#     return f"hp:{s}|lr={lr}|ep={ep}|bs={bs}"


# @dataclass
# class Selection:
#     row: pd.Series
#     pool_idx: np.ndarray
#     robust_score: float
#     local_mad: float
#     rank_gap: float
#     scope: str
#     fallback_used: bool
#     fallback_reason: str
#     chosen_key: str = ""
#     cycles_seen: str = ""
#     # consistency diagnostics
#     cons_hits: float = np.nan
#     cons_cycles_seen: float = np.nan
#     cons_hit_rate: float = np.nan
#     cons_median_val: float = np.nan
#     cons_median_robust: float = np.nan
#     cons_val_iqr: float = np.nan


# def _basic_select(df_group: pd.DataFrame) -> Selection:
#     df = _prep_numeric(df_group)
#     if C["val"] not in df.columns:
#         raise ValueError(f"Missing required column: {C['val']}")

#     df = df.loc[np.isfinite(df[C["val"]].to_numpy(dtype=float))].reset_index(drop=True)
#     if len(df) < 5:
#         i = int(df[C["val"]].idxmin())
#         r = df.loc[i]
#         return Selection(
#             row=r,
#             pool_idx=np.array([], dtype=int),
#             robust_score=float(r[C["val"]]),
#             local_mad=np.nan,
#             rank_gap=np.nan,
#             scope="none",
#             fallback_used=True,
#             fallback_reason="too_few_rows_after_cleaning",
#             chosen_key=_cand_key(r),
#         )

#     pool = pick_pool_idx(df)
#     if len(pool) == 0:
#         i = int(df[C["val"]].idxmin())
#         r = df.loc[i]
#         return Selection(
#             row=r,
#             pool_idx=np.array([], dtype=int),
#             robust_score=float(r[C["val"]]),
#             local_mad=np.nan,
#             rank_gap=np.nan,
#             scope="none",
#             fallback_used=True,
#             fallback_reason="no_candidates",
#             chosen_key=_cand_key(r),
#         )

#     scores, scopes, mads, rgs = compute_robust_scores(df, pool)
#     cand_scores = scores.loc[pool]
#     if cand_scores.isna().all():
#         i = int(df[C["val"]].idxmin())
#         r = df.loc[i]
#         return Selection(
#             row=r,
#             pool_idx=pool,
#             robust_score=float(r[C["val"]]),
#             local_mad=np.nan,
#             rank_gap=np.nan,
#             scope="none",
#             fallback_used=True,
#             fallback_reason="robust_scores_all_nan",
#             chosen_key=_cand_key(r),
#         )

#     best_i = int(cand_scores.idxmin())
#     r = df.loc[best_i]
#     return Selection(
#         row=r,
#         pool_idx=pool,
#         robust_score=float(scores.loc[best_i]),
#         local_mad=float(mads.loc[best_i]),
#         rank_gap=float(rgs.loc[best_i]),
#         scope=str(scopes.loc[best_i]) or "within_strategy",
#         fallback_used=False,
#         fallback_reason="",
#         chosen_key=_cand_key(r),
#     )


# def _consistency_select(df_group: pd.DataFrame) -> Selection:
#     df0 = _prep_numeric(df_group)
#     if C["cycle"] not in df0.columns:
#         return _basic_select(df0)

#     df0 = df0.loc[np.isfinite(df0[C["val"]].to_numpy(dtype=float))].reset_index(drop=True)
#     cycles = sorted([int(x) for x in pd.unique(df0[C["cycle"]]) if pd.notna(x)])
#     if len(cycles) <= 1:
#         return _basic_select(df0)

#     hits: Dict[str, int] = {}
#     seen: Dict[str, int] = {}
#     per_val: Dict[str, List[float]] = {}
#     per_rob: Dict[str, List[float]] = {}

#     for cyc in cycles:
#         df_c = df0.loc[df0[C["cycle"]] == cyc].reset_index(drop=True)
#         if len(df_c) < 3:
#             continue

#         pool = pick_pool_idx(df_c)
#         if len(pool) == 0:
#             continue

#         scores, _, _, _ = compute_robust_scores(df_c, pool)
#         cand_scores = scores.loc[pool].dropna()
#         if len(cand_scores) == 0:
#             continue

#         top_k = int(min(CONS["topk_per_cycle"], len(cand_scores)))
#         top_idx = cand_scores.nsmallest(top_k).index.to_numpy(dtype=int)

#         # seen for every candidate in pool for that cycle
#         for i in pool.astype(int):
#             row = df_c.loc[int(i)]
#             k = _cand_key(row)
#             seen[k] = seen.get(k, 0) + 1
#             per_val.setdefault(k, []).append(float(row[C["val"]]))
#             per_rob.setdefault(k, []).append(float(scores.loc[int(i)]) if pd.notna(scores.loc[int(i)]) else np.nan)

#         # hits for top-k
#         for i in top_idx:
#             row = df_c.loc[int(i)]
#             k = _cand_key(row)
#             hits[k] = hits.get(k, 0) + 1

#     if not seen:
#         return _basic_select(df0)

#     rows = []
#     for k, n_seen in seen.items():
#         h = hits.get(k, 0)
#         hr = h / n_seen if n_seen > 0 else 0.0
#         vals = np.array(per_val.get(k, []), dtype=float)
#         robs = np.array(per_rob.get(k, []), dtype=float)

#         med_val = float(np.nanmedian(vals)) if np.isfinite(vals).any() else np.nan
#         med_rob = float(np.nanmedian(robs)) if np.isfinite(robs).any() else np.nan
#         v_iqr = iqr(vals)
#         tie = med_rob + CONS["stab_lambda"] * (v_iqr if np.isfinite(v_iqr) else 0.0)

#         rows.append(
#             dict(
#                 key=k,
#                 hits=int(h),
#                 cycles_seen=int(n_seen),
#                 hit_rate=float(hr),
#                 median_val=med_val,
#                 median_robust=med_rob,
#                 val_iqr=float(v_iqr) if np.isfinite(v_iqr) else np.nan,
#                 tie_score=float(tie) if np.isfinite(tie) else np.inf,
#             )
#         )

#     cand_tbl = pd.DataFrame(rows)
#     filt = cand_tbl.copy()
#     if CONS["min_hits"] is not None:
#         filt = filt.loc[filt["hits"] >= int(CONS["min_hits"])]
#     if CONS["min_rate"] is not None:
#         filt = filt.loc[filt["hit_rate"] >= float(CONS["min_rate"])]

#     if len(filt) == 0:
#         filt = cand_tbl.copy()

#     filt = filt.sort_values(["hits", "tie_score", "median_val"], ascending=[False, True, True], na_position="last").reset_index(drop=True)
#     best_key = str(filt.loc[0, "key"])

#     # choose concrete row instance from pooled df0
#         # --- choose concrete row instance from pooled df0 (FIXED) ---
#     pool0 = pick_pool_idx(df0)
#     if len(pool0) == 0:
#         return _basic_select(df0)  # defensive

#     scores0, scopes0, mads0, rgs0 = compute_robust_scores(df0, pool0)

#     # Only materialize among candidates that are actually in the global pool0,
#     # otherwise robust_score will be NaN because compute_robust_scores only fills pool indices.
#     keys_pool = df0.loc[pool0].apply(_cand_key, axis=1)
#     idx_key_pool = pool0[(keys_pool == best_key).to_numpy()]

#     # If best_key never appears inside the global pool, do NOT force it.
#     # This matches the intended "pool-first" logic and avoids NaNs.
#     if len(idx_key_pool) == 0:
#         return _basic_select(df0)

#     sc = scores0.loc[idx_key_pool]
#     # sc should be non-NaN for these indices (they're in pool0), but keep it safe:
#     sc = sc.dropna()
#     if len(sc) == 0:
#         return _basic_select(df0)

#     chosen_i = int(sc.idxmin())
#     row = df0.loc[chosen_i]

#     return Selection(
#         row=row,
#         pool_idx=pool0,
#         robust_score=float(scores0.loc[chosen_i]),
#         local_mad=float(mads0.loc[chosen_i]) if pd.notna(mads0.loc[chosen_i]) else np.nan,
#         rank_gap=float(rgs0.loc[chosen_i]) if pd.notna(rgs0.loc[chosen_i]) else np.nan,
#         scope=str(scopes0.loc[chosen_i]) or "within_strategy",
#         fallback_used=False,
#         fallback_reason="",
#         chosen_key=best_key,
#         cycles_seen=str(cycles),
#         cons_hits=float(filt.loc[0, "hits"]),
#         cons_cycles_seen=float(filt.loc[0, "cycles_seen"]),
#         cons_hit_rate=float(filt.loc[0, "hit_rate"]),
#         cons_median_val=float(filt.loc[0, "median_val"]),
#         cons_median_robust=float(filt.loc[0, "median_robust"]),
#         cons_val_iqr=float(filt.loc[0, "val_iqr"]) if pd.notna(filt.loc[0, "val_iqr"]) else np.nan,
#     )



# def select_champion(df_group: pd.DataFrame) -> Selection:
#     has_cycles = C["cycle"] in df_group.columns and df_group[C["cycle"]].nunique(dropna=True) > 1
#     if CONS["use"] and has_cycles:
#         return _consistency_select(df_group)
#     return _basic_select(df_group)


# # =========================
# # Audit (TEST-only, no influence)
# # =========================
# def audit_pool(df_clean: pd.DataFrame, pool_idx: np.ndarray, chosen_row: pd.Series) -> Dict[str, float]:
#     out = dict(
#         pool_size=float(len(pool_idx)),
#         pool_best_test=np.nan,
#         pool_pct_good_test=np.nan,
#         pool_chosen_test_percentile=np.nan,
#         val_test_spearman=np.nan,
#         test_good_threshold=np.nan,
#     )

#     if C["test"] not in df_clean.columns or len(pool_idx) == 0:
#         return out

#     v = to_num(df_clean[C["val"]])
#     t = to_num(df_clean[C["test"]])

#     m = np.isfinite(v.to_numpy()) & np.isfinite(t.to_numpy())
#     if m.sum() >= 3:
#         out["val_test_spearman"] = float(pd.Series(v[m]).corr(pd.Series(t[m]), method="spearman"))

#     t_pool = t.loc[pool_idx].to_numpy(dtype=float)
#     t_pool = t_pool[np.isfinite(t_pool)]
#     if len(t_pool) == 0:
#         return out

#     out["pool_best_test"] = float(np.min(t_pool))

#     t_all = t.to_numpy(dtype=float)
#     t_all = t_all[np.isfinite(t_all)]
#     if len(t_all) == 0:
#         return out

#     thr = TEST_AUDIT["good_abs"] if TEST_AUDIT["good_abs"] is not None else float(np.quantile(t_all, TEST_AUDIT["good_q"]))
#     out["test_good_threshold"] = float(thr)
#     out["pool_pct_good_test"] = float((t_pool <= thr).mean())

#     chosen_test = float(pd.to_numeric(chosen_row.get(C["test"], np.nan), errors="coerce"))
#     if np.isfinite(chosen_test):
#         out["pool_chosen_test_percentile"] = float((t_pool <= chosen_test).mean())

#     return out


# def render_summary(summary: pd.DataFrame) -> None:
#     if summary is None or len(summary) == 0:
#         print("\nNo rows to show in summary.")
#         return

#     df = summary.copy()
#     rename = {
#         "dataset": "Dataset",
#         "well": "Well",
#         "architecture": "Arch",
#         "chosen_strategy": "Chosen strategy",
#         "chosen_trial": "Chosen trial",
#         "chosen_key": "Chosen key",
#         "chosen_val": "Chosen VAL",
#         "chosen_test": "Chosen TEST",
#         "best_test": "Best TEST",
#         "regret_test": "TEST regret",
#         "ratio_test": "TEST ratio",
#         "robust_score": "Robust score",
#         "val_test_spearman": "Spearman(VAL,TEST)",
#         "pool_chosen_test_percentile": "Chosen TEST pct in pool",
#         "cycles_seen": "Cycles",
#     }
#     keep = [c for c in rename.keys() if c in df.columns]
#     df = df[keep].rename(columns=rename)

#     round_map = {
#         "Chosen VAL": 4,
#         "Chosen TEST": 4,
#         "Best TEST": 4,
#         "TEST regret": 4,
#         "TEST ratio": 3,
#         "Robust score": 4,
#         "Spearman(VAL,TEST)": 3,
#         "Chosen TEST pct in pool": 3,
#     }
#     for col, nd in round_map.items():
#         if col in df.columns:
#             df[col] = pd.to_numeric(df[col], errors="coerce").round(nd)

#     print("\n=== Summary (TEST is audit-only) ===")
#     if not _maybe_display_dataframe(df):
#         print(df.to_string(index=False))


# def render_quick_audit(summary: pd.DataFrame) -> None:
#     if summary is None or len(summary) == 0:
#         return

#     def q(x: np.ndarray, p: float) -> float:
#         x = x[np.isfinite(x)]
#         return float(np.quantile(x, p)) if len(x) else np.nan

#     regret = pd.to_numeric(summary.get("regret_test", np.nan), errors="coerce").to_numpy(dtype=float)
#     ratio = pd.to_numeric(summary.get("ratio_test", np.nan), errors="coerce").to_numpy(dtype=float)
#     spearman = pd.to_numeric(summary.get("val_test_spearman", np.nan), errors="coerce").to_numpy(dtype=float)

#     print("\n=== Quick audit (campaign-wide; TEST is audit-only) ===")
#     print("Spearman(VAL,TEST) is a rank correlation (range [-1, +1]). +1 means VAL ranking matches TEST ranking (good proxy); 0 means weak/no relationship; -1 means VAL ranking is inverted vs TEST (risky proxy).")
#     print("TEST regret = chosen_TEST - best_TEST (absolute gap; 0 is perfect; lower is better). TEST ratio = chosen_TEST / best_TEST (relative gap; 1.0 is perfect; e.g., 1.21 means the chosen model is ~21% worse than the best TEST for that group).")
#     print("Quantiles: median (p50) is the typical case; p90 means 90% of groups are at or below that value (so it's a 'near-worst-case' summary).\n")

#     print(f"- TEST regret: median={q(regret, 0.5):.4g} | p90={q(regret, 0.9):.4g} | max={np.nanmax(regret) if np.isfinite(regret).any() else np.nan:.6g}")
#     print(f"- TEST ratio : median={q(ratio, 0.5):.4g} | p90={q(ratio, 0.9):.4g} | max={np.nanmax(ratio) if np.isfinite(ratio).any() else np.nan:.6g}")
#     print(f"- Spearman(VAL,TEST): median={q(spearman, 0.5):.4g} | min={np.nanmin(spearman) if np.isfinite(spearman).any() else np.nan:.6g} | max={np.nanmax(spearman) if np.isfinite(spearman).any() else np.nan:.6g}")

#     n_bad = int(np.sum(np.isfinite(spearman) & (spearman < 0)))
#     if n_bad:
#         print(f"\n⚠️  Note: {n_bad}/{len(summary)} groups have negative Spearman. In those groups, VAL ranking may be a poor proxy for TEST ranking.")


# # =========================
# # Main
# # =========================
# def run_campaign() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     base = Path(CAMPAIGN_DIR)
#     master, diag = build_master_leaderboard(base)

#     req = {C["dataset"], C["well"], C["arch"]}
#     if not req.issubset(master.columns):
#         raise RuntimeError("Master leaderboard missing group columns for (dataset, well, architecture).")

#     groups = list(master.groupby([C["dataset"], C["well"], C["arch"]], dropna=False))
#     if CONSOLE:
#         CONSOLE.print(RICH_PANEL(f"Groups found: {len(groups)}", title="Grouping", expand=False))
#     else:
#         print(f"\nGroups found: {len(groups)}")

#     rows: List[Dict[str, object]] = []

#     for (dataset, well, arch), df_g in groups[:MAX_GROUPS]:
#         dataset, well, arch = str(dataset), str(well), str(arch)
#         if C["val"] not in df_g.columns:
#             print(f"[skip] {dataset} | {well} | {arch}: missing {C['val']}")
#             continue

#         try:
#             sel = select_champion(df_g)
#         except Exception as e:
#             df_tmp = _prep_numeric(df_g)
#             s = to_num(df_tmp[C["val"]])
#             i = int(s.idxmin())
#             r = df_tmp.loc[i]
#             sel = Selection(
#                 row=r,
#                 pool_idx=np.array([], dtype=int),
#                 robust_score=float(s.min()),
#                 local_mad=np.nan,
#                 rank_gap=np.nan,
#                 scope="none",
#                 fallback_used=True,
#                 fallback_reason=f"selector_error:{type(e).__name__}",
#                 chosen_key=_cand_key(r),
#             )

#         # audit inputs should match selection cleaning
#         df_clean = _prep_numeric(df_g)
#         df_clean = df_clean.loc[np.isfinite(df_clean[C["val"]].to_numpy(dtype=float))].reset_index(drop=True)
#         pool_for_audit = pick_pool_idx(df_clean)
#         aud = audit_pool(df_clean, pool_for_audit, sel.row)

#         chosen_val = float(pd.to_numeric(sel.row.get(C["val"], np.nan), errors="coerce"))
#         chosen_test = float(pd.to_numeric(sel.row.get(C["test"], np.nan), errors="coerce")) if C["test"] in df_g.columns else np.nan
#         best_test = float(to_num(_prep_numeric(df_g)[C["test"]]).min()) if C["test"] in df_g.columns else np.nan
#         regret = (chosen_test - best_test) if (np.isfinite(chosen_test) and np.isfinite(best_test)) else np.nan
#         ratio = (chosen_test / best_test) if (np.isfinite(chosen_test) and np.isfinite(best_test) and best_test != 0) else np.nan

#         cyc = sorted([int(x) for x in pd.unique(df_g.get(C["cycle"], pd.Series([], dtype=float))) if pd.notna(x)])
#         cycles_seen = sel.cycles_seen or (str(cyc) if cyc else "")

#         rows.append(
#             dict(
#                 dataset=dataset,
#                 well=well,
#                 architecture=arch,
#                 family=short_family(dataset),
#                 pool_method=POOL_METHOD.lower(),
#                 chosen_strategy=sel.row.get(C["strat"], ""),
#                 chosen_trial=sel.row.get(C["trial"], np.nan),
#                 chosen_key=sel.chosen_key or _cand_key(sel.row),
#                 chosen_val=chosen_val,
#                 chosen_test=chosen_test,
#                 best_test=best_test,
#                 regret_test=regret,
#                 ratio_test=ratio,
#                 robust_score=float(sel.robust_score),
#                 local_mad=float(sel.local_mad) if np.isfinite(sel.local_mad) else np.nan,
#                 rank_gap=float(sel.rank_gap) if np.isfinite(sel.rank_gap) else np.nan,
#                 neighbor_scope=sel.scope,
#                 fallback_used=bool(sel.fallback_used),
#                 fallback_reason=sel.fallback_reason,
#                 n_rows=int(len(df_g)),
#                 n_candidates=int(len(sel.pool_idx)) if sel.pool_idx is not None else np.nan,
#                 cycles_seen=cycles_seen,
#                 # consistency diagnostics
#                 cons_hits=sel.cons_hits,
#                 cons_cycles_seen=sel.cons_cycles_seen,
#                 cons_hit_rate=sel.cons_hit_rate,
#                 cons_median_val=sel.cons_median_val,
#                 cons_median_robust=sel.cons_median_robust,
#                 cons_val_iqr=sel.cons_val_iqr,
#                 # audit pool
#                 pool_size=aud.get("pool_size", np.nan),
#                 pool_best_test=aud.get("pool_best_test", np.nan),
#                 pool_pct_good_test=aud.get("pool_pct_good_test", np.nan),
#                 pool_chosen_test_percentile=aud.get("pool_chosen_test_percentile", np.nan),
#                 val_test_spearman=aud.get("val_test_spearman", np.nan),
#                 test_good_threshold=aud.get("test_good_threshold", np.nan),
#             )
#         )

#         # Plot (optional, safe)
#         if PLOT and plot_error_distributions_story is not None:
#             extra = f"robust={sel.robust_score:.3g} | scope={sel.scope} | pool={POOL_METHOD.lower()}"
#             if POOL_METHOD.lower() == "val_band":
#                 extra += f" | band={POOL_CFG['drop']:.2f}-{(POOL_CFG['drop'] + POOL_CFG['take']):.2f}"
#             if CONS["use"] and np.isfinite(sel.cons_hits):
#                 extra += f" | CONS hits={int(sel.cons_hits)}/{int(sel.cons_cycles_seen) if np.isfinite(sel.cons_cycles_seen) else 0}"
#                 extra += f" | medR={sel.cons_median_robust:.3g}" if np.isfinite(sel.cons_median_robust) else ""
#             if sel.fallback_used:
#                 extra += " | FALLBACK"
#             if cycles_seen:
#                 extra += f" | cycles={cycles_seen}"

#             palette = _get_color_palette("default") if _get_color_palette is not None else None
#             plot_error_distributions_story(
#                 df_g,
#                 chosen_row=sel.row,
#                 title=str(extra),
#                 dataset=str(dataset),
#                 well=str(well),
#                 architecture=str(arch),
#                 chosen_strategy=str(sel.row.get(C["strat"], "")),
#                 chosen_trial=sel.row.get(C["trial"], None),
#                 best_test=best_test,
#                 regret_test=regret,
#                 ratio_test=ratio,
#                 spearman_val_test=aud.get("val_test_spearman", None),
#                 pool_chosen_test_percentile=aud.get("pool_chosen_test_percentile", None),
#                 palette=palette,
#                 show=True,
#             )

#     summary = pd.DataFrame(rows).sort_values("regret_test", ascending=False, na_position="last").reset_index(drop=True)

#     render_summary(summary)
#     render_quick_audit(summary)

#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     if SAVE_MASTER:
#         p = OUTPUT_DIR / MASTER_NAME
#         master.to_csv(p, index=False)
#         print(f"\n✅ master saved: {p} ({len(master)} rows)")
#     if SAVE_SUMMARY:
#         p = OUTPUT_DIR / SUMMARY_NAME
#         summary.to_csv(p, index=False)
#         print(f"✅ summary saved: {p} ({len(summary)} rows)")
#     if SAVE_DIAGNOSTICS:
#         p = OUTPUT_DIR / DIAGNOSTICS_NAME
#         diag.to_csv(p, index=False)
#         print(f"✅ diagnostics saved: {p} ({len(diag)} rows)")

#     return summary, master, diag


# if __name__ == "__main__":
#     run_campaign()


In [ ]:
# # ============================================================
# # HPO Campaign Sweeper: compare POOL_METHOD = "val_band" vs "top_pct"
# # Assumptions:
# #   - The "robust selector" code (run_campaign + globals like CAMPAIGN_DIR, POOL_METHOD, PLOT, etc.)
# #     is already defined in the notebook kernel (previous cell).
# #   - This cell does NOT modify that code permanently (it snapshots & restores globals).
# # Output:
# #   - Prints a clean, readable comparison table + win counts (no files saved here).
# # ============================================================

# from __future__ import annotations

# import io
# import os
# import math
# import contextlib
# from pathlib import Path
# from typing import Dict, Any, Tuple, Optional, List

# import numpy as np
# import pandas as pd


# # ----------------------------
# # Settings (edit if needed)
# # ----------------------------
# BASE_DIR = Path("/home/gabriel/Documentos/Equinor/src/experiment_configs")
# CAMPAIGN_PREFIX = "HPO_"
# METHODS = ["val_band", "top_pct"]

# # If you want to limit for a quick test:
# MAX_CAMPAIGNS: Optional[int] = None  # e.g. 3

# # A "winner" composite score to rank methods per-campaign (lower is better).
# # Keep it simple and transparent:
# #   score = median_regret + p90_regret
# USE_COMPOSITE_SCORE = True

# # ----------------------------
# # Pretty printing helpers
# # ----------------------------
# def _maybe_display(df: pd.DataFrame) -> bool:
#     try:
#         from IPython.display import display
#         display(df)
#         return True
#     except Exception:
#         return False

# def _q(x: np.ndarray, p: float) -> float:
#     x = x[np.isfinite(x)]
#     return float(np.quantile(x, p)) if len(x) else np.nan

# def _fmt(x: Any, nd: int = 4) -> Any:
#     try:
#         if x is None:
#             return np.nan
#         xf = float(x)
#         if not np.isfinite(xf):
#             return np.nan
#         return round(xf, nd)
#     except Exception:
#         return x

# def _safe_bool_in_globals(name: str) -> bool:
#     return name in globals() and isinstance(globals()[name], bool)


# # ----------------------------
# # Campaign discovery
# # ----------------------------
# def discover_campaigns(base_dir: Path, prefix: str = "HPO_") -> List[Path]:
#     if not base_dir.is_dir():
#         raise FileNotFoundError(f"Base dir not found: {base_dir}")
#     camps = []
#     for p in sorted(base_dir.iterdir()):
#         if p.is_dir() and p.name.startswith(prefix):
#             # heuristic: campaign should have results/ directory
#             if (p / "results").is_dir():
#                 camps.append(p)
#     return camps


# # ----------------------------
# # Metrics extraction
# # ----------------------------
# def summarize_campaign_run(summary: pd.DataFrame) -> Dict[str, float]:
#     """
#     summary: the per-group summary DF returned by run_campaign()
#     Returns campaign-level metrics for method comparison.
#     """
#     if summary is None or len(summary) == 0:
#         return {
#             "n_groups": 0,
#             "med_regret": np.nan,
#             "p90_regret": np.nan,
#             "max_regret": np.nan,
#             "med_ratio": np.nan,
#             "p90_ratio": np.nan,
#             "max_ratio": np.nan,
#             "med_spearman": np.nan,
#             "min_spearman": np.nan,
#             "neg_spearman_rate": np.nan,
#         }

#     regret = pd.to_numeric(summary.get("regret_test", np.nan), errors="coerce").to_numpy(dtype=float)
#     ratio  = pd.to_numeric(summary.get("ratio_test", np.nan), errors="coerce").to_numpy(dtype=float)
#     spear  = pd.to_numeric(summary.get("val_test_spearman", np.nan), errors="coerce").to_numpy(dtype=float)

#     n_groups = int(len(summary))

#     out = {
#         "n_groups": n_groups,
#         "med_regret": _q(regret, 0.50),
#         "p90_regret": _q(regret, 0.90),
#         "max_regret": float(np.nanmax(regret)) if np.isfinite(regret).any() else np.nan,
#         "med_ratio":  _q(ratio, 0.50),
#         "p90_ratio":  _q(ratio, 0.90),
#         "max_ratio":  float(np.nanmax(ratio)) if np.isfinite(ratio).any() else np.nan,
#         "med_spearman": _q(spear, 0.50),
#         "min_spearman": float(np.nanmin(spear)) if np.isfinite(spear).any() else np.nan,
#         "neg_spearman_rate": float(np.mean((spear < 0) & np.isfinite(spear))) if np.isfinite(spear).any() else np.nan,
#     }
#     return out


# # ----------------------------
# # Runner: execute existing run_campaign() while swapping globals
# # ----------------------------
# def run_existing_selector_for_campaign(
#     campaign_dir: Path,
#     pool_method: str,
#     *,
#     plot: bool = False,
#     silence: bool = True,
# ) -> Tuple[Optional[pd.DataFrame], Dict[str, Any]]:
#     """
#     Uses the already-defined run_campaign() + globals from the previous cell.
#     Temporarily overrides CAMPAIGN_DIR, POOL_METHOD, PLOT, and (optionally) SAVE_* flags.
#     Restores all afterwards.

#     Returns: (summary_df, run_meta)
#     """
#     if "run_campaign" not in globals() or not callable(globals()["run_campaign"]):
#         raise RuntimeError("run_campaign() not found in globals(). Make sure the selector cell ran successfully.")

#     # Snapshot relevant globals (only those that exist)
#     keys_to_override = [
#         "CAMPAIGN_DIR", "POOL_METHOD", "PLOT",
#         "SAVE_MASTER", "SAVE_SUMMARY", "SAVE_DIAGNOSTICS",
#         "MAX_GROUPS_TO_PLOT", "MAX_GROUPS",  # depending on which version you have
#     ]
#     snapshot = {k: globals().get(k, None) for k in keys_to_override if k in globals()}

#     # Apply overrides (only if the symbol exists in your selector cell)
#     if "CAMPAIGN_DIR" in globals(): globals()["CAMPAIGN_DIR"] = str(campaign_dir)
#     if "POOL_METHOD" in globals(): globals()["POOL_METHOD"] = str(pool_method)
#     if "PLOT" in globals(): globals()["PLOT"] = bool(plot)

#     # Strongly recommended for sweeping: disable saving to avoid IO + clutter
#     if _safe_bool_in_globals("SAVE_MASTER"): globals()["SAVE_MASTER"] = False
#     if _safe_bool_in_globals("SAVE_SUMMARY"): globals()["SAVE_SUMMARY"] = False
#     if _safe_bool_in_globals("SAVE_DIAGNOSTICS"): globals()["SAVE_DIAGNOSTICS"] = False

#     # Also: avoid limiting groups by plot cap if your code uses that
#     # (do nothing unless it exists; leaving as-is is fine)
#     # if "MAX_GROUPS_TO_PLOT" in globals(): globals()["MAX_GROUPS_TO_PLOT"] = 999999

#     buf_out, buf_err = io.StringIO(), io.StringIO()
#     try:
#         if silence:
#             with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
#                 summary, *_ = globals()["run_campaign"]()
#         else:
#             summary, *_ = globals()["run_campaign"]()

#         meta = {
#             "stdout": buf_out.getvalue(),
#             "stderr": buf_err.getvalue(),
#             "campaign_dir": str(campaign_dir),
#             "pool_method": str(pool_method),
#             "ok": True,
#         }
#         return summary, meta

#     except Exception as e:
#         meta = {
#             "stdout": buf_out.getvalue(),
#             "stderr": buf_err.getvalue(),
#             "campaign_dir": str(campaign_dir),
#             "pool_method": str(pool_method),
#             "ok": False,
#             "error": f"{type(e).__name__}: {e}",
#         }
#         return None, meta

#     finally:
#         # Restore snapshot
#         for k, v in snapshot.items():
#             globals()[k] = v


# # ----------------------------
# # Main sweep
# # ----------------------------
# campaigns = discover_campaigns(BASE_DIR, prefix=CAMPAIGN_PREFIX)
# if MAX_CAMPAIGNS is not None:
#     campaigns = campaigns[: int(MAX_CAMPAIGNS)]

# print(f"Found {len(campaigns)} campaigns under: {BASE_DIR}")

# all_rows: List[Dict[str, Any]] = []
# errors: List[Dict[str, Any]] = []

# for camp in campaigns:
#     for method in METHODS:
#         summary_df, meta = run_existing_selector_for_campaign(
#             camp, method, plot=False, silence=True
#         )
#         if not meta.get("ok", False):
#             errors.append(meta)
#             all_rows.append({
#                 "campaign": camp.name,
#                 "method": method,
#                 "ok": False,
#                 "error": meta.get("error", "unknown"),
#             })
#             continue

#         metrics = summarize_campaign_run(summary_df)
#         row = {
#             "campaign": camp.name,
#             "method": method,
#             "ok": True,
#             **metrics,
#         }

#         if USE_COMPOSITE_SCORE:
#             # Lower is better; NaNs stay NaN
#             row["score_regret_med_p90"] = (
#                 float(metrics["med_regret"]) + float(metrics["p90_regret"])
#                 if np.isfinite(metrics["med_regret"]) and np.isfinite(metrics["p90_regret"])
#                 else np.nan
#             )

#         all_rows.append(row)

# raw = pd.DataFrame(all_rows)

# # ----------------------------
# # Build comparison (wide table)
# # ----------------------------
# ok = raw[raw["ok"] == True].copy()
# if len(ok) == 0:
#     print("\nNo successful runs. Errors:")
#     if errors:
#         print(pd.DataFrame(errors)[["campaign_dir", "pool_method", "error"]].to_string(index=False))
#     raise RuntimeError("Sweep finished with zero successful results.")

# # Pivot metrics wide: columns like med_regret_val_band, med_regret_top_pct, ...
# metrics_cols = [c for c in ok.columns if c not in ("campaign", "method", "ok")]
# wide = (
#     ok.pivot(index="campaign", columns="method", values=metrics_cols)
#       .sort_index()
# )

# # Flatten MultiIndex columns
# wide.columns = [f"{m}_{meth}" for (m, meth) in wide.columns]
# wide = wide.reset_index()

# def _delta(col: str) -> None:
#     # delta = top_pct - val_band (negative => top_pct better; positive => val_band better)
#     a = f"{col}_top_pct"
#     b = f"{col}_val_band"
#     if a in wide.columns and b in wide.columns:
#         wide[f"delta_{col}_(top_pct - val_band)"] = wide[a] - wide[b]

# for c in ["med_regret", "p90_regret", "max_regret", "med_ratio", "p90_ratio", "max_ratio", "med_spearman", "neg_spearman_rate"]:
#     _delta(c)

# # Winner columns (for the key decision metrics)
# def _winner_lower_is_better(metric: str) -> pd.Series:
#     a = wide.get(f"{metric}_top_pct")
#     b = wide.get(f"{metric}_val_band")
#     out = []
#     for x, y in zip(a, b):
#         if not (np.isfinite(x) or np.isfinite(y)):
#             out.append("")
#         elif np.isfinite(x) and not np.isfinite(y):
#             out.append("top_pct")
#         elif np.isfinite(y) and not np.isfinite(x):
#             out.append("val_band")
#         else:
#             out.append("top_pct" if x < y else ("val_band" if y < x else "tie"))
#     return pd.Series(out)

# def _winner_higher_is_better(metric: str) -> pd.Series:
#     a = wide.get(f"{metric}_top_pct")
#     b = wide.get(f"{metric}_val_band")
#     out = []
#     for x, y in zip(a, b):
#         if not (np.isfinite(x) or np.isfinite(y)):
#             out.append("")
#         elif np.isfinite(x) and not np.isfinite(y):
#             out.append("top_pct")
#         elif np.isfinite(y) and not np.isfinite(x):
#             out.append("val_band")
#         else:
#             out.append("top_pct" if x > y else ("val_band" if y > x else "tie"))
#     return pd.Series(out)

# wide["winner_med_regret"] = _winner_lower_is_better("med_regret")
# wide["winner_p90_regret"] = _winner_lower_is_better("p90_regret")
# wide["winner_max_regret"] = _winner_lower_is_better("max_regret")
# wide["winner_med_ratio"]  = _winner_lower_is_better("med_ratio")
# wide["winner_med_spearman"] = _winner_higher_is_better("med_spearman")

# if USE_COMPOSITE_SCORE and "score_regret_med_p90_top_pct" in wide.columns and "score_regret_med_p90_val_band" in wide.columns:
#     wide["winner_score_regret_med_p90"] = _winner_lower_is_better("score_regret_med_p90")

# # Formatting for display
# display_cols = [
#     "campaign",
#     "n_groups_val_band", "n_groups_top_pct",
#     "med_regret_val_band", "med_regret_top_pct", "delta_med_regret_(top_pct - val_band)", "winner_med_regret",
#     "p90_regret_val_band", "p90_regret_top_pct", "delta_p90_regret_(top_pct - val_band)", "winner_p90_regret",
#     "max_regret_val_band", "max_regret_top_pct", "delta_max_regret_(top_pct - val_band)", "winner_max_regret",
#     "med_ratio_val_band", "med_ratio_top_pct", "delta_med_ratio_(top_pct - val_band)", "winner_med_ratio",
#     "med_spearman_val_band", "med_spearman_top_pct", "delta_med_spearman_(top_pct - val_band)", "winner_med_spearman",
#     "neg_spearman_rate_val_band", "neg_spearman_rate_top_pct",
# ]
# if USE_COMPOSITE_SCORE and "score_regret_med_p90_val_band" in wide.columns:
#     display_cols += [
#         "score_regret_med_p90_val_band", "score_regret_med_p90_top_pct",
#         "delta_score_regret_med_p90_(top_pct - val_band)",
#         "winner_score_regret_med_p90",
#     ]

# display_cols = [c for c in display_cols if c in wide.columns]
# disp = wide[display_cols].copy()

# # Round numeric cols for readability
# for c in disp.columns:
#     if c == "campaign":
#         continue
#     if pd.api.types.is_numeric_dtype(disp[c]):
#         # slightly different rounding for rates
#         nd = 3 if ("rate" in c or "spearman" in c or "ratio" in c) else 4
#         disp[c] = pd.to_numeric(disp[c], errors="coerce").round(nd)

# # Sort by composite score winner / deltas if available
# if USE_COMPOSITE_SCORE and "delta_score_regret_med_p90_(top_pct - val_band)" in disp.columns:
#     # Most negative means top_pct better; most positive means val_band better
#     disp = disp.sort_values("delta_score_regret_med_p90_(top_pct - val_band)", ascending=True, na_position="last")
# elif "delta_p90_regret_(top_pct - val_band)" in disp.columns:
#     disp = disp.sort_values("delta_p90_regret_(top_pct - val_band)", ascending=True, na_position="last")

# print("\n=== HPO Sweep: Method Comparison (val_band vs top_pct) ===")
# if not _maybe_display(disp):
#     print(disp.to_string(index=False))

# # ----------------------------
# # Win counts summary
# # ----------------------------
# def _count_wins(col: str) -> Dict[str, int]:
#     s = wide[col].value_counts(dropna=False).to_dict()
#     return {k: int(s.get(k, 0)) for k in ["val_band", "top_pct", "tie", ""]}

# summary_rows = []
# for col in ["winner_med_regret", "winner_p90_regret", "winner_max_regret", "winner_med_ratio", "winner_med_spearman"]:
#     wins = _count_wins(col)
#     summary_rows.append({"criterion": col.replace("winner_", ""), **wins})

# if USE_COMPOSITE_SCORE and "winner_score_regret_med_p90" in wide.columns:
#     wins = _count_wins("winner_score_regret_med_p90")
#     summary_rows.append({"criterion": "score_regret_med_p90", **wins})

# wins_df = pd.DataFrame(summary_rows)

# print("\n=== Win counts across campaigns ===")
# if not _maybe_display(wins_df):
#     print(wins_df.to_string(index=False))

# # ----------------------------
# # Errors (if any)
# # ----------------------------
# if len(raw[raw["ok"] == False]) > 0:
#     err_tbl = raw[raw["ok"] == False][["campaign", "method", "error"]].copy()
#     print("\n=== Errors (some runs failed) ===")
#     if not _maybe_display(err_tbl):
#         print(err_tbl.to_string(index=False))
